# HeatShield AI — Training Pipeline (Kaggle)

**Single notebook** — setup → ingest → train → export.

**Kaggle Secrets** (Add-ons → Secrets):
- `CDSAPI_KEY` — ERA5 ingest (optional)
- `TMD_API_KEY` — TMD ingest (optional)

**Workflow:**
1. SETUP — clone repo (public), install deps
2. INGEST — NASA POWER data
3. VALIDATE — row counts per station
4. STAGE 1 — h=24 sweep (all stations, CUDA P100, 60 trials)
5. STAGE 2 — refine weak slots (200 trials)
6. STAGE 3 — h=6/12 for ready stations (optional)
7. EXPORT — ZIP to `/kaggle/working/` (download from Output tab)

## 1) Setup — Clone, Install, Paths

In [ ]:
import hashlib
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

# --- optional Kaggle secrets (CDSAPI_KEY, TMD_API_KEY) ---
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    for _key in ("CDSAPI_KEY", "TMD_API_KEY"):
        try:
            os.environ[_key] = _secrets.get_secret(_key)
        except Exception:
            pass  # optional — skip if not set
except ImportError:
    pass  # not on Kaggle

# --- repo constants ---
REPO      = "/kaggle/working/Heat-wave-backend"
GH_OWNER  = "orbitorls"
GH_REPO   = "HeatShield"
GH_BRANCH = "main"

# --- clone (public repo, no token needed) ---
if not Path(f"{REPO}/.git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth=1", f"--branch={GH_BRANCH}",
         f"https://github.com/{GH_OWNER}/{GH_REPO}.git", REPO],
        check=True,
    )
else:
    print("Repo already cloned — pulling latest")
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)

# --- install deps (skip if requirements unchanged) ---
REQ       = f"{REPO}/requirements-train.txt"
HASH_FILE = f"{REPO}/.kaggle_req_hash"
new_hash  = hashlib.sha256(Path(REQ).read_bytes()).hexdigest()
old_hash  = Path(HASH_FILE).read_text().strip() if Path(HASH_FILE).exists() else ""
if new_hash == old_hash:
    print("requirements-train.txt unchanged — skip pip install")
else:
    print("Installing requirements-train.txt ...")
    subprocess.run(["pip", "install", "-q", "-r", REQ], check=True)
    Path(HASH_FILE).write_text(new_hash)
    print("Install done.")

# --- python path ---
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print("cwd:", os.getcwd())

# --- working directories (all under /kaggle/working/) ---
WORK_ROOT    = Path("/kaggle/working/heatshield")
MODEL_DIR    = WORK_ROOT / "models" / "forecast_v3"
VERSIONS_DIR = WORK_ROOT / "models" / "forecast_versions"
EXPORTS_DIR  = WORK_ROOT / "exports"
for _d in (MODEL_DIR, VERSIONS_DIR, EXPORTS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# --- symlink repo's model dir → persistent working dir ---
LOCAL_MODELS = Path(REPO) / "app" / "models" / "forecast_v3"
LOCAL_MODELS.parent.mkdir(parents=True, exist_ok=True)
if LOCAL_MODELS.exists() and not LOCAL_MODELS.is_symlink():
    for _item in LOCAL_MODELS.iterdir():
        _target = MODEL_DIR / _item.name
        if not _target.exists():
            shutil.move(str(_item), str(_target))
    shutil.rmtree(str(LOCAL_MODELS))
if not LOCAL_MODELS.exists():
    LOCAL_MODELS.symlink_to(str(MODEL_DIR))
print("v3 models →", MODEL_DIR)

# --- GPU auto-detection ---
def _detect_device() -> str:
    try:
        r = subprocess.run(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=5,
        )
        if r.returncode == 0 and r.stdout.strip():
            print(f"GPU detected: {r.stdout.strip().splitlines()[0]} → device=cuda")
            return "cuda"
    except Exception:
        pass
    print("No GPU detected → device=cpu")
    return "cpu"

DEVICE    = _detect_device()
N_WORKERS = 1 if DEVICE == "cuda" else max(2, min(4, (os.cpu_count() or 2) // 2))
print(f"DEVICE={DEVICE}  N_WORKERS={N_WORKERS}")

# --- session state ---
RUN_IDS_CREATED: list[str] = []
_RUN_ID_SET:     set[str]  = set()
TRAIN_TIMINGS:   list[dict] = []

print("Setup complete.")

In [ ]:
# Import + GPU check
for mod in ("numpy", "pandas", "xgboost", "lightgbm", "sklearn", "optuna"):
    try:
        m = importlib.import_module(mod)
        print(f"{mod:<12s} {getattr(m, '__version__', '?')}")
    except ImportError as exc:
        print(f"{mod:<12s} MISSING ({exc})")
try:
    import torch
    print(f"torch        {torch.__version__} | cuda={torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"             device: {torch.cuda.get_device_name(0)}")
except Exception:
    print("torch        not installed")

from app.data.stations import STATIONS
assert set(STATIONS.keys()) == {"BKK_01", "CNX_01", "KKN_01", "HYI_01", "RYG_01"}
print("Stations:", list(STATIONS.keys()))

## 2) Ingest — NASA POWER (2021-01-01 → today)

In [ ]:
import datetime

START = "2021-01-01"
END   = datetime.date.today().isoformat()
print(f"Ingesting {START} → {END}")

subprocess.run(
    ["python", "scripts/ingest_nasa_power.py", "--start", START, "--end", END],
    cwd=REPO,
    check=True,
)
print("Ingest done.")

## 3) Validate — min 500 rows per station

In [ ]:
from datetime import date as _date
from app.data.loaders import read_observations

start_d = _date.fromisoformat(START)
end_d   = _date.fromisoformat(END)
MIN_ROWS = 500
bad: list[tuple] = []

for sid in STATIONS:
    n = len(read_observations(sid, start_d, end_d))
    status = "OK" if n >= MIN_ROWS else "FAIL"
    print(f"  [{status}] {sid}: {n} rows")
    if n < MIN_ROWS:
        bad.append((sid, n))

if bad:
    raise RuntimeError(f"Not enough data for training: {bad}")
print("Data readiness: OK")

## 4) Training Helper

In [ ]:
import time

LOG_DIR   = Path(REPO) / "logs" / "train"
RUNS_ROOT = Path(REPO) / "logs" / "eval" / "runs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

_help_text = subprocess.check_output(
    ["python", "scripts/train_forecast.py", "--help"],
    text=True, stderr=subprocess.STDOUT, cwd=REPO,
)


def _has(flag: str) -> bool:
    return flag in _help_text


def _workers(device: str, requested: int | None) -> int:
    if requested is not None and requested > 0:
        return requested
    # GPU: single process to avoid contention
    if (device or "").lower() in {"gpu", "cuda"}:
        return 1
    return max(1, min(4, (os.cpu_count() or 2) // 2))


def run_train(
    *,
    trials: int,
    station: str | None = None,
    horizons: str | None = None,
    run_tag: str = "run",
    workers: int | None = None,
    backend: str = "lightgbm",
    device: str = "cuda",
    gate_backend: str = "lightgbm",
    force: bool = False,
) -> tuple[str, Path]:
    run_id = f"{run_tag}_{datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
    nw = _workers(device, workers)

    cmd = [
        "python", "-u", "scripts/train_forecast.py",
        "--trials", str(trials),
        "--start",  START,
        "--end",    END,
    ]
    if _has("--run-id"):        cmd += ["--run-id",       run_id]
    if _has("--device"):        cmd += ["--device",       device]
    if _has("--backend"):       cmd += ["--backend",      backend]
    if _has("--gate-backend"):  cmd += ["--gate-backend", gate_backend]
    if _has("--workers") and nw > 1: cmd += ["--workers", str(nw)]
    if station:  cmd += ["--station",  station]
    if horizons: cmd += ["--horizons", horizons]
    if force and _has("--force"): cmd += ["--force"]

    log_path = LOG_DIR / f"{run_id}.log"
    print(f"\nCmd: {' '.join(cmd)}")
    print(f"Log: {log_path}")

    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    t0 = time.perf_counter()
    with open(log_path, "w", encoding="utf-8") as _f:
        _proc = subprocess.Popen(
            cmd, cwd=REPO,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, env=env,
        )
        for _line in _proc.stdout:
            print(_line, end="")
            _f.write(_line)
        rc = _proc.wait()

    elapsed = time.perf_counter() - t0
    if rc != 0:
        raise RuntimeError(f"Training failed rc={rc} — see {log_path}")

    if run_id not in _RUN_ID_SET:
        _RUN_ID_SET.add(run_id)
        RUN_IDS_CREATED.append(run_id)
    TRAIN_TIMINGS.append({"run_id": run_id, "tag": run_tag,
                           "seconds": round(elapsed, 2), "workers": nw})
    print(f"\nDone {run_id} in {elapsed:.1f}s")
    return run_id, log_path


print("run_train() ready.")

## 5) Stage 1 — h=24 Sweep (all stations, 60 trials)

- GPU auto-detected at setup — ใช้ CUDA ถ้ามี, fallback เป็น CPU อัตโนมัติ
- Champion-challenger: model ใหม่ save เฉพาะเมื่อ test MAE ดีขึ้น

In [ ]:
stage1_run_id, _ = run_train(
    trials=60,
    horizons="24",
    run_tag="stage1",
    device=DEVICE,
    workers=N_WORKERS,
    gate_backend="lightgbm",
)
print("Stage 1:", stage1_run_id)

## 6) Stage 2 — Refine Weak Slots (200 more trials)

อ่าน leaderboard จาก Stage 1 — retrain เฉพาะ `not_ready` / `candidate` เท่านั้น

In [ ]:
import json
from collections import defaultdict

lb_path = RUNS_ROOT / stage1_run_id / "leaderboard.json"
if not lb_path.exists():
    raise FileNotFoundError(f"leaderboard.json not found: {lb_path}")

rows = json.loads(lb_path.read_text(encoding="utf-8"))
weak_by_station: dict[str, set[int]] = defaultdict(set)
for r in rows:
    status = str(r.get("status", "")).lower()
    if status in {"not_ready", "candidate"} or r.get("skill_score") is None:
        sid, h = r.get("station"), r.get("horizon_h")
        if sid and h is not None:
            weak_by_station[str(sid)].add(int(h))

print(f"Weak: {sum(len(v) for v in weak_by_station.values())} / {len(rows)} slots")
for sid, hs in sorted(weak_by_station.items()):
    print(f"  {sid}: h={sorted(hs)}")

for sid in sorted(weak_by_station):
    hz = ",".join(str(h) for h in sorted(weak_by_station[sid]))
    print(f"\nRefining {sid} h={hz}")
    run_train(
        trials=200, station=sid, horizons=hz,
        run_tag=f"stage2_{sid}", device=DEVICE, workers=N_WORKERS,
        gate_backend="lightgbm", force=True,
    )

print("\nStage 2 complete.")

## 7) Stage 3 — h=6/12 for Ready Stations (optional)

ข้ามถ้าต้องการแค่ h=24

In [ ]:
all_rows: list[dict] = []
for run_id in RUN_IDS_CREATED:
    lb = RUNS_ROOT / run_id / "leaderboard.json"
    if lb.exists():
        all_rows.extend(json.loads(lb.read_text("utf-8")))

latest: dict[tuple, dict] = {}
for r in all_rows:
    key = (r.get("station"), r.get("horizon_h"))
    latest[key] = r

ready_stations = sorted({
    sid for (sid, h), r in latest.items()
    if h == 24 and str(r.get("status", "")).lower() == "ready"
})
print(f"Ready (h=24): {ready_stations}")

for sid in ready_stations:
    print(f"\nTraining h=6,12 for {sid}")
    run_train(
        trials=120, station=sid, horizons="6,12",
        run_tag=f"stage3_{sid}", device=DEVICE, workers=N_WORKERS,
        gate_backend="lightgbm", force=False,
    )

print("\nStage 3 complete.")

## 8) Export — Snapshot ZIP

ZIP จะอยู่ที่ `/kaggle/working/` — ดาวน์โหลดได้จาก **Output tab** ของ notebook นี้

In [ ]:
import zipfile

if not RUN_IDS_CREATED:
    raise RuntimeError("No runs in this session. Train first, then export.")

t0 = time.perf_counter()


def _next_version(base: Path) -> int:
    versions = [
        int(d.name[1:]) for d in base.iterdir()
        if d.is_dir() and d.name.startswith("v") and d.name[1:].isdigit()
    ]
    return (max(versions) + 1) if versions else 1


slot_meta: dict[tuple, dict] = {}
snap_warnings: list[str] = []

for run_id in RUN_IDS_CREATED:
    lb = RUNS_ROOT / run_id / "leaderboard.json"
    if not lb.exists():
        snap_warnings.append(f"missing_leaderboard:{run_id}")
        continue
    for r in json.loads(lb.read_text(encoding="utf-8")):
        sid, h = r.get("station"), r.get("horizon_h")
        if sid in (None, "all") or h is None:
            continue
        key = (str(sid), int(h))
        slot_meta[key] = {
            "station": str(sid), "horizon_h": int(h),
            "backend": r.get("backend"), "status": r.get("status"),
            "source_run_id": run_id,
        }

if not slot_meta:
    raise RuntimeError("No slots found in leaderboards.")

snap_ver = f"v{_next_version(VERSIONS_DIR)}"
snap_dir = VERSIONS_DIR / snap_ver
snap_dir.mkdir(parents=True, exist_ok=False)

copied_files = 0
copied_bytes = 0
for (sid, h) in sorted(slot_meta):
    src = LOCAL_MODELS / sid / f"h{h}"
    dst = snap_dir / sid / f"h{h}"
    if not src.exists():
        snap_warnings.append(f"missing_model:{sid}:h{h}")
        continue
    shutil.copytree(src, dst, dirs_exist_ok=True)
    for _fp in dst.rglob("*"):
        if _fp.is_file():
            copied_files += 1
            copied_bytes += _fp.stat().st_size

cm = LOCAL_MODELS / "choice_matrix.json"
if cm.exists():
    shutil.copy2(cm, snap_dir / "choice_matrix.json")
else:
    snap_warnings.append("missing_choice_matrix")

manifest = {
    "created_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "snapshot_version": snap_ver,
    "run_ids": RUN_IDS_CREATED,
    "slots": [slot_meta[k] for k in sorted(slot_meta)],
    "copied_files": copied_files,
    "copied_bytes": copied_bytes,
    "warnings": snap_warnings,
    "timings": TRAIN_TIMINGS,
}
(snap_dir / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))

stamp    = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
zip_name = f"HeatShield_artifacts_{snap_ver}_{stamp}.zip"
# Output to /kaggle/working/ so it appears in the Output tab
zip_path = Path("/kaggle/working") / zip_name

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for fp in sorted(snap_dir.rglob("*")):
        if fp.is_file():
            arc = Path("models") / "forecast_versions" / snap_ver / fp.relative_to(snap_dir)
            zf.write(fp, arc.as_posix())
    for run_id in RUN_IDS_CREATED:
        run_dir = RUNS_ROOT / run_id
        if run_dir.exists():
            for fp in sorted(run_dir.rglob("*")):
                if fp.is_file():
                    arc = Path("eval") / "runs" / run_id / fp.relative_to(run_dir)
                    zf.write(fp, arc.as_posix())
    zf.writestr("manifest.json", json.dumps(manifest, indent=2, ensure_ascii=False))

elapsed = time.perf_counter() - t0
print(f"Snapshot: {snap_dir}")
print(f"ZIP:      {zip_path} ({zip_path.stat().st_size / 1024 / 1024:.1f} MB)")
print(f"Slots:    {len(slot_meta)}")
print(f"Elapsed:  {elapsed:.1f}s")
print(f"\nDownload from: Kaggle Output tab → {zip_name}")
if snap_warnings:
    print("Warnings:")
    for w in snap_warnings:
        print(" -", w)

## 9) Local Evaluation (Windows — หลัง download ZIP)

In [ ]:
print(r"""
LOCAL EVALUATION (PowerShell in d:\Heat-wave-backend):

1. Download ZIP จาก Kaggle:
   Notebook → Output tab → HeatShield_artifacts_v*.zip

2. Extract:
   $ZIP = Get-ChildItem .\HeatShield_artifacts_v*.zip | Sort LastWriteTime -Desc | Select -First 1
   $TMP = ".\artifact_unpack"
   Remove-Item $TMP -Recurse -Force -ErrorAction SilentlyContinue
   Expand-Archive $ZIP.FullName -DestinationPath $TMP -Force

3. Promote snapshot:
   $VER = Get-ChildItem "$TMP\models\forecast_versions" -Directory | Sort Name -Desc | Select -First 1
   Remove-Item app\models\forecast_v3 -Recurse -Force -ErrorAction SilentlyContinue
   New-Item -ItemType Directory -Force -Path app\models\forecast_v3 | Out-Null
   Copy-Item "$($VER.FullName)\*" "app\models\forecast_v3\" -Recurse -Force

4. Verify:
   python -c "from app.ml.registry import load_latest_v3; print(load_latest_v3('BKK_01', 24).backend_name)"

5. Evaluate:
   python scripts/evaluate_model.py

6. Smoke test:
   uvicorn app.main:app --reload
""")

## 10) Latest Leaderboard

In [ ]:
try:
    runs = sorted(RUNS_ROOT.iterdir(), key=lambda p: p.name, reverse=True)
    latest_run = runs[0] if runs else None
except FileNotFoundError:
    latest_run = None

if latest_run:
    lb_md = latest_run / "leaderboard.md"
    if lb_md.exists():
        print(f"Run: {latest_run.name}\n")
        print(lb_md.read_text())
    else:
        print(f"No leaderboard.md for {latest_run.name}")
else:
    print("No runs found yet.")